# X-ray attenuation and image contrast

*medical · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/medical/xray_attenuation)

X-rays fade exponentially through the body, $I = I_0\,e^{-\int\mu\,dl}$. A chest phantom made of lung, soft tissue, bone and iodine contrast is projected ray by ray with one einsum, showing why bone stands out at low energy and how a real tube spectrum hardens with depth.

**Vector form**

$$I(E) = I_0 \exp\!\left(-\int_{\text{ray}} \mu(E, \mathbf r)\, dl\right)$$

**Index (tensor) form**

$$L_{mx} = M_{myx}\,\Delta l_y \qquad -\ln\frac{I_{ex}}{I_0} = \mu_{em}\,L_{mx} \qquad \frac{I_x}{I_0} = S_e\, e^{-\mu_{em} L_{mx}}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $M_{myx}$ | `masks` | (M, Y, X) | one-hot phantom: 1 where pixel $(y, x)$ is material $m$ |
| $\Delta l_y$ | `dl` | (Y,) | path length of a ray inside pixel row $y$ |
| $L_{mx}$ | `einsum('myx,y->mx', masks, dl)` | (M, X) | repeated $y$ ⇒ summed: length of material $m$ crossed by ray $x$ |
| $\mu_{em}$ | `mu` | (E, M) | linear attenuation coefficient of material $m$ at energy $E_e$ (1/cm) |
| $\mu_{em} L_{mx}$ | `einsum('em,mx->ex', mu, L)` | (E, X) | repeated $m$ ⇒ summed: line integral $\int \mu\, dl$ for every energy and ray |
| $S_e$ | `spectrum` | (E,) | tube spectrum (Kramers law, aluminium filter), $\sum_e S_e = 1$ |
| $S_e\, e^{-\mu_{em} L_{mx}}$ | `einsum('e,ex->x', S, T)` | (X,) | repeated $e$ ⇒ summed: transmitted fraction of a polychromatic beam |

In log space the projection is additive: $-\ln(I/I_0)$ is the sum of $\mu\,L$ over materials, so a monochromatic image is a linear map of the material lengths. A homogeneous slab halves the beam every half-value layer $\ln 2/\mu$. Photoelectric absorption grows roughly like $Z^3/E^3$, so bone contrast is large at 30–50 keV and fades at higher energy; iodine jumps above its K-edge at 33 keV, which is why it is used as a contrast agent. A real tube emits a spectrum: soft photons are absorbed first, the mean energy rises and the effective $\mu$ drops with depth (beam hardening). Simplifications: straight parallel rays, no scatter, no detector response, eight tabulated energies with rounded $\mu$ values. All numbers are approximate and for education only, not for clinical use.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/medical/xray_attenuation/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    e = tensors.energies(dev)  # (E,) keV
    mu = tensors.attenuation_table(params["iodine_mg_ml"], dev)  # (E, M)
    mu_chosen = tensors.interpolate_loglog(e, mu, params["energy_kev"])  # (M,)
    x, y, masks = tensors.phantom(params["n_pixels"], dev)  # masks (M, Y, X)
    dl = tensors.path_weights(y)  # (Y,)
    spectrum = tensors.tube_spectrum(params["kvp"], e)  # (E,)

    # Table energies plus the chosen energy as the last row.
    mu_all = torch.cat([mu, mu_chosen[None]])  # (E+1, M)
    e_all = torch.cat([e, torch.tensor([float(params["energy_kev"])], device=dev)])
    lengths, mono, poly = solve.projections(mu, mu_chosen, masks, dl, spectrum)
    mu_maps = torch.einsum("em,myx->eyx", mu_all, masks)  # μ(y, x) at every energy
    profiles = torch.stack([solve.ray_profiles(m, masks, dl) for m in mu_all])  # (E+1, Y, X)
    contrast = solve.contrast_vs_energy(mu)
    hardening = solve.beam_hardening(mu[:, 2], spectrum, e)

    centre = len(x) // 2
    print(
        f"energy {params['energy_kev']:.0f} keV:  μ(tissue) = {mu_chosen[2]:.3f} /cm, μ(bone) = {mu_chosen[3]:.3f} /cm"
    )
    print(f"half-value layer in soft tissue: {solve.half_value_layer(float(mu_chosen[2])):.2f} cm")
    print(
        f"central ray crosses {', '.join(f'{n} {lengths[m, centre]:.1f} cm' for m, n in enumerate(tensors.MATERIALS))}"
    )
    print(f"central ray: I/I0 = {mono[-1, centre]:.2e} (mono), {poly[centre]:.2e} ({params['kvp']:.0f} kVp spectrum)")
    for k, ek in enumerate(e.tolist()):
        c = contrast[k]
        print(f"  {ek:5.0f} keV  contrast  lung {c[0]:+.3f}  bone {c[1]:+.3f}  iodine {c[2]:+.3f}")
    d, _, mu_eff, e_mean = hardening
    print(
        f"beam hardening: μ_eff {mu_eff[0]:.3f} → {mu_eff[-1]:.3f} /cm, "
        f"mean energy {e_mean[0]:.0f} → {e_mean[-1]:.0f} keV"
    )

    depth = y - y[0] + (y[1] - y[0]) / 2
    return plot.animate(
        x, depth, mu_maps, profiles, mono, poly, e_all, contrast, hardening, out_dir / "xray_attenuation.gif"
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/medical/xray_attenuation/equation.py
r"""Physics → tensor translation.

The Beer–Lambert law: a narrow X-ray beam of energy E loses photons exponentially,

    I = I_0 \exp\left(-\int \mu(E, \mathbf r)\, dl\right).

The phantom is a stack of material masks M_{myx} (1 where pixel (y, x) is material m).
A vertical ray at column x crosses a length L_{mx} of each material:

    L_{mx} = M_{myx} \Delta l_y                  (y summed: path-length weights)

and the line integral of μ splits into "how much of each material × how strongly it attenuates":

    -\ln(I_{ex}/I_0) = \mu_{em} L_{mx}            (m summed)

Indices: e = photon energy, m = material, y = depth along the ray, x = ray (detector pixel).
A polychromatic beam with spectrum S_e is a weighted sum of monochromatic beams:

    I_x / I_0 = S_e \exp(-\mu_{em} L_{mx})        (e summed, S_e sums to 1)
"""

import torch


def path_lengths(masks: torch.Tensor, dl: torch.Tensor) -> torch.Tensor:
    r"""L_{mx} = M_{myx} \Delta l_y.   masks: (M, Y, X), dl: (Y,) → (M, X) in cm"""
    return torch.einsum("myx,y->mx", masks, dl)


def line_integral(mu: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    r"""\int \mu\,dl = \mu_{em} L_{mx}.   mu: (E, M), lengths: (M, X) → (E, X), dimensionless"""
    return torch.einsum("em,mx->ex", mu, lengths)


def transmission(mu: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    r"""Beer–Lambert I/I_0 = \exp(-\mu_{em} L_{mx}) for every energy and ray: (E, X)."""
    return torch.exp(-line_integral(mu, lengths))


def polychromatic(spectrum: torch.Tensor, mu: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    r"""I_x/I_0 = S_e \exp(-\mu_{em} L_{mx}):  spectrum (E,) sums to 1 → (X,)."""
    return torch.einsum("e,ex->x", spectrum, transmission(mu, lengths))


def contrast(mu_insert: torch.Tensor, mu_background: torch.Tensor, thickness: float) -> torch.Tensor:
    r"""Subject contrast of an insert of thickness t replacing background: C = 1 - e^{-(\mu_i - \mu_b) t}."""
    return 1.0 - torch.exp(-(mu_insert - mu_background) * thickness)


def effective_mu(depth: torch.Tensor, intensity: torch.Tensor) -> torch.Tensor:
    r"""Local effective attenuation \mu_{eff}(d) = -\,d\ln I / dd (finite differences): (D-1,)."""
    return -torch.diff(torch.log(intensity)) / torch.diff(depth)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/medical/xray_attenuation/tensors.py
"""Build the concrete tensors: attenuation table μ_{em}, the phantom masks M_{myx} and the tube spectrum S_e.

All attenuation coefficients are rounded values in 1/cm (derived from NIST XCOM mass attenuation
coefficients × typical densities). They are good enough to see the trends, not for clinical use.
"""

import torch

ENERGIES_KEV = [30.0, 40.0, 50.0, 60.0, 80.0, 100.0, 120.0, 150.0]
MATERIALS = ["air", "lung", "soft tissue", "bone", "iodine contrast"]

# Linear attenuation coefficients μ (1/cm) at the energies above.
MU_TABLE = {
    "air": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],  # ~2e-4 1/cm: negligible
    "lung": [0.100, 0.072, 0.061, 0.055, 0.049, 0.046, 0.043, 0.040],  # inflated lung, ρ ≈ 0.26 g/cm³
    "soft tissue": [0.398, 0.284, 0.241, 0.218, 0.195, 0.181, 0.171, 0.160],  # ρ ≈ 1.06 g/cm³
    "bone": [2.56, 1.28, 0.81, 0.60, 0.43, 0.36, 0.32, 0.28],  # cortical bone, ρ ≈ 1.92 g/cm³
}
# Mass attenuation coefficient of iodine (cm²/g); the K-edge at 33.2 keV makes it jump between 30 and 40 keV.
IODINE_MASS_MU = [8.7, 22.1, 12.3, 7.6, 3.5, 1.94, 1.2, 0.69]
# Aluminium tube filter (1/cm), removes the softest photons from the spectrum.
ALUMINIUM_MU = [3.05, 1.53, 0.99, 0.75, 0.545, 0.46, 0.42, 0.372]


def energies(device: torch.device) -> torch.Tensor:
    """Photon energies E_e in keV: shape (E,)."""
    return torch.tensor(ENERGIES_KEV, device=device)


def attenuation_table(iodine_mg_ml: float, device: torch.device) -> torch.Tensor:
    """μ_{em} of shape (E, M) in 1/cm. The contrast agent is soft tissue (blood) plus dissolved iodine."""
    columns = [MU_TABLE[m] for m in MATERIALS[:-1]]
    iodine = torch.tensor(IODINE_MASS_MU) * (iodine_mg_ml / 1000.0)  # (cm²/g)·(g/cm³) = 1/cm
    columns.append((torch.tensor(MU_TABLE["soft tissue"]) + iodine).tolist())
    return torch.tensor(columns, device=device).T  # (E, M)


def interpolate_loglog(e_table: torch.Tensor, mu: torch.Tensor, energy: float) -> torch.Tensor:
    """μ_m at one energy, interpolated linearly in log(μ) vs log(E) (μ = 0 stays 0): (E, M) → (M,)."""
    log_e = torch.log(e_table)
    target = torch.log(torch.tensor(float(energy), device=e_table.device))
    i = int(torch.searchsorted(log_e, target).clamp(1, len(e_table) - 1))  # table interval [i-1, i]
    w = (target - log_e[i - 1]) / (log_e[i] - log_e[i - 1])
    lo, hi = mu[i - 1].clamp_min(1e-12), mu[i].clamp_min(1e-12)
    out = torch.exp((1 - w) * torch.log(lo) + w * torch.log(hi))
    return torch.where(mu[i] > 0, out, torch.zeros_like(out))


def phantom(n_pixels: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """A crude chest slice, 32 cm wide and 24 cm deep.

    Returns
      x (X,), y (Y,) pixel centres in cm, and masks M_{myx} of shape (M, Y, X): one-hot material per pixel.
    The beam travels along y (front to back), so each column x is one ray.
    """
    nx, ny = n_pixels, (3 * n_pixels) // 4
    x = (torch.arange(nx, device=device) + 0.5) * (32.0 / nx) - 16.0
    y = (torch.arange(ny, device=device) + 0.5) * (24.0 / ny) - 12.0
    yy, xx = torch.meshgrid(y, x, indexing="ij")  # (Y, X)

    def ellipse(cx, cy, ax, ay):
        return ((xx - cx) / ax) ** 2 + ((yy - cy) / ay) ** 2 <= 1.0

    label = torch.zeros(ny, nx, dtype=torch.long, device=device)  # 0 = air
    label[ellipse(0.0, 0.0, 15.0, 10.0)] = 2  # body: soft tissue
    label[ellipse(-6.5, 0.5, 3.8, 6.5)] = 1  # lungs
    label[ellipse(6.5, 0.5, 3.8, 6.5)] = 1
    label[ellipse(-1.0, 2.0, 3.0, 3.0)] = 4  # heart filled with iodinated blood
    label[ellipse(0.0, -6.5, 1.8, 1.8)] = 3  # spine
    label[ellipse(0.0, 8.6, 1.6, 0.6)] = 3  # sternum
    masks = torch.nn.functional.one_hot(label, len(MATERIALS)).permute(2, 0, 1).float()  # (M, Y, X)
    return x, y, masks


def path_weights(y: torch.Tensor) -> torch.Tensor:
    """Δl_y: length of the ray inside each pixel row (cm). Straight vertical rays → the pixel height: (Y,)."""
    return torch.full_like(y, float(y[1] - y[0]))


def tube_spectrum(kvp: float, e: torch.Tensor, filter_cm: float = 0.25) -> torch.Tensor:
    """S_e: Kramers' law (kVp − E) for the bremsstrahlung, × an aluminium filter, × bin width; sums to 1."""
    kramers = (kvp - e).clamp_min(0.0)
    filtered = kramers * torch.exp(-torch.tensor(ALUMINIUM_MU, device=e.device) * filter_cm)
    weights = filtered * torch.gradient(e)[0]  # bins are not equally wide
    return weights / weights.sum()


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/medical/xray_attenuation/solve.py
"""Evaluate the attenuation law: projection images, depth profiles of each ray, contrast and beam hardening."""

import torch

from . import equation


def projections(
    mu: torch.Tensor, mu_chosen: torch.Tensor, masks: torch.Tensor, dl: torch.Tensor, spectrum: torch.Tensor
):
    """Return (L (M, X), monochromatic I/I0 at the table energies + the chosen one (E+1, X),
    polychromatic I/I0 (X,))."""
    lengths = equation.path_lengths(masks, dl)  # (M, X)
    mono = equation.transmission(torch.cat([mu, mu_chosen[None]]), lengths)  # (E+1, X)
    poly = equation.polychromatic(spectrum, mu, lengths)  # (X,)
    return lengths, mono, poly


def ray_profiles(mu_at_energy: torch.Tensor, masks: torch.Tensor, dl: torch.Tensor) -> torch.Tensor:
    r"""I(y)/I_0 inside the body for every ray: exp(-\sum_{y' \le y} \mu(y', x) \Delta l_{y'}):  (Y, X)."""
    mu_map = torch.einsum("m,myx->yx", mu_at_energy, masks)  # μ(y, x) at one energy
    return torch.exp(-torch.cumsum(mu_map * dl[:, None], dim=0))


def contrast_vs_energy(mu: torch.Tensor, thickness: float = 1.0) -> torch.Tensor:
    """Contrast of a 1 cm insert of lung, bone and iodine contrast against soft tissue: (E, 3)."""
    soft = mu[:, 2:3]  # (E, 1)
    inserts = mu[:, [1, 3, 4]]  # lung, bone, iodine contrast
    return equation.contrast(inserts, soft, thickness)


def beam_hardening(mu_soft: torch.Tensor, spectrum: torch.Tensor, e: torch.Tensor, max_depth: float = 30.0):
    """A polychromatic beam through a growing slab of soft tissue.

    Returns (depth (D,), I/I0 (D,), μ_eff (D-1,), mean energy of the transmitted photons (D,)).
    """
    depth = torch.linspace(0.0, max_depth, 121, device=e.device)
    survived = spectrum[None, :] * torch.exp(-depth[:, None] * mu_soft[None, :])  # (D, E) photons left per bin
    intensity = survived.sum(1)
    mean_energy = torch.einsum("de,e->d", survived, e) / intensity
    return depth, intensity, equation.effective_mu(depth, intensity), mean_energy


def half_value_layer(mu: float) -> float:
    """Thickness that halves a monochromatic beam: HVL = ln 2 / μ."""
    return float(torch.log(torch.tensor(2.0)) / mu)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/medical/xray_attenuation/plot.py
"""Animate: rays sweep across the phantom and build the projection profile, then the energy is swept."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import LogNorm
from matplotlib.ticker import NullLocator

from problems._common.plotting import ACCENT, frame_indices, save_gif

SWEEP, ENERGY_SWEEP = 40, 24  # logical steps of the two phases


def animate(
    x: torch.Tensor,
    depth: torch.Tensor,
    mu_maps: torch.Tensor,  # (E+1, Y, X): table energies, then the chosen energy last
    profiles: torch.Tensor,  # (E+1, Y, X) I(y)/I0 inside the body
    mono: torch.Tensor,  # (E+1, X) projection at each energy
    poly: torch.Tensor,  # (X,)
    e_all: torch.Tensor,  # (E+1,)
    contrast: torch.Tensor,  # (E, 3)
    hardening: tuple,
    path: Path,
) -> Path:
    x, depth = x.cpu().numpy(), depth.cpu().numpy()
    mu_maps, profiles = mu_maps.cpu().numpy(), profiles.cpu().numpy()
    mono, poly, e_all, contrast = mono.cpu().numpy(), poly.cpu().numpy(), e_all.cpu().numpy(), contrast.cpu().numpy()
    d_slab, i_slab, mu_eff, e_mean = (h.cpu().numpy() for h in hardening)
    n_e = len(e_all) - 1
    chosen = n_e  # index of the user-chosen energy

    fig, axes = plt.subplots(2, 2, figsize=(9, 5.6))
    (ax_img, ax_prof), (ax_con, ax_bh) = axes
    extent = (x[0], x[-1], depth[-1], depth[0])
    img = ax_img.imshow(mu_maps[chosen], cmap=plt.get_cmap("bone", 24), vmin=0, vmax=0.8, extent=extent, aspect="auto")
    fig.colorbar(img, ax=ax_img, label=r"$\mu$ (1/cm)", fraction=0.05)
    ax_img.set_xlabel("x (cm)")
    ax_img.set_ylabel("depth along beam (cm)")
    ax_img.grid(False)
    rays = ax_img.scatter([], [], c=[], s=5, cmap="autumn", norm=LogNorm(1e-3, 1))  # colour = I/I0 along the ray
    title = ax_img.set_title("")

    ax_prof.set_xlim(x[0], x[-1])
    ax_prof.set_ylim(1e-4, 1.5)
    ax_prof.set_yscale("log")
    ax_prof.yaxis.set_minor_locator(NullLocator())  # fewer tick lines → faster frames
    ax_prof.set_xlabel("detector position x (cm)")
    ax_prof.set_ylabel(r"$I/I_0$")
    (prof_line,) = ax_prof.plot([], [], color=ACCENT[0], lw=1.8, label="monochromatic")
    (poly_line,) = ax_prof.plot([], [], color=ACCENT[1], lw=1.2, ls="--", label="tube spectrum")
    ax_prof.legend(loc="lower center", fontsize=8)
    ax_prof.set_title("projection image (one row)")

    names = ["lung", "bone", "iodine"]
    for k in range(3):
        ax_con.plot(e_all[:n_e], contrast[:, k], "o-", ms=3, color=ACCENT[k + 1], label=names[k])
    ax_con.axhline(0, color="#9ca3af", lw=0.8)
    e_mark = ax_con.axvline(e_all[chosen], color="#f59e0b", lw=1.5)
    ax_con.set_xlabel("photon energy (keV)")
    ax_con.set_ylabel("contrast vs tissue (1 cm)")
    ax_con.legend(fontsize=8, loc="upper right")

    ax_bh.plot(d_slab[1:], mu_eff, color=ACCENT[0], lw=1.8)
    ax_bh.set_xlabel("soft-tissue thickness (cm)")
    ax_bh.set_ylabel(r"$\mu_{\rm eff}$ (1/cm)", color=ACCENT[0])
    ax_bh.set_title("beam hardening")
    twin = ax_bh.twinx()
    twin.plot(d_slab, e_mean, color=ACCENT[4], lw=1.4, ls="--")
    twin.set_ylabel("mean energy (keV)", color=ACCENT[4])
    twin.grid(False)

    ny = len(depth)
    ray_rows = np.arange(0, ny, 2)

    def update(step: int):
        if step < SWEEP:  # phase 1: rays sweep across x at the chosen energy
            e_idx = chosen
            col = min(len(x) - 1, round(step * (len(x) - 1) / (SWEEP - 1)))
            prof_line.set_data(x[: col + 1], mono[e_idx, : col + 1])
            poly_line.set_data([], [])
            cols = [c for c in range(col - 8, col + 1, 4) if c >= 0]
        else:  # phase 2: sweep the photon energy over the table
            e_idx = min(n_e - 1, (step - SWEEP) * n_e // ENERGY_SWEEP)
            prof_line.set_data(x, mono[e_idx])
            poly_line.set_data(x, poly)
            cols = list(range(4, len(x), max(1, len(x) // 10)))
        img.set_data(mu_maps[e_idx])
        rx = np.repeat(x[cols], len(ray_rows))
        ry = np.tile(depth[ray_rows], len(cols))
        rays.set_offsets(np.column_stack([rx, ry]))
        rays.set_array(profiles[e_idx][np.ix_(ray_rows, cols)].T.ravel())
        e_mark.set_xdata([e_all[e_idx], e_all[e_idx]])
        title.set_text(f"phantom at E = {e_all[e_idx]:.0f} keV (beam ↓)")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(SWEEP + ENERGY_SWEEP, 64), path, fps=8)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'energy_kev': 60,
    'kvp': 120,
    'iodine_mg_ml': 10,
    'n_pixels': 128
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.medical.xray_attenuation")]:
    del sys.modules[name]
main = importlib.import_module("problems.medical.xray_attenuation.main")
problem_dir = Path("problems/medical/xray_attenuation")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
